# 05 - ANN & LSTM Benchmark

I kept deep learning as a benchmark. A small LSTM uses the previous 24 hourly observations.

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn

hourly = pd.read_csv("../01_Data/processed/hourly_energy.csv", parse_dates=["datetime"])
y = hourly["energy_kwh"].dropna().values.astype("float32")

In [ ]:
train_end = int(len(y) * 0.8)
mu = y[:train_end].mean()
sd = y[:train_end].std()
y_scaled = (y - mu) / sd

seq_len = 24
X, target, pos = [], [], []
for i in range(seq_len, len(y_scaled)):
    X.append(y_scaled[i-seq_len:i])
    target.append(y_scaled[i])
    pos.append(i)

X = np.array(X, dtype="float32")[:, :, None]
target = np.array(target, dtype="float32")[:, None]

In [ ]:
class SmallLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 16, batch_first=True)
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1])

In [ ]:
model = SmallLSTM()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

train_mask = np.array(pos) < train_end
for epoch in range(80):
    optimizer.zero_grad()
    out = model(torch.tensor(X[train_mask]))
    loss = loss_fn(out, torch.tensor(target[train_mask]))
    loss.backward()
    optimizer.step()

print("LSTM benchmark finished")